In [1]:
# Importing libraries & frameworks 
import random
import genesis as gs
import numpy as np
import quaternion
import cv2
import json
from pathlib import Path
from trimesh.transformations import quaternion_from_euler
from scipy.spatial.transform import Rotation as R


[I 04/17/25 17:27:54.451 22112] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [ ]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

bottle_radius = 0.025
bottle_pos = [0, 0.0, 0.1] 

plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl", 
        pos=bottle_pos,            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)
initial_gripper_pos = [-0.5, 0.0, 0.10]
initial_gripper_quat = [0.707, 0.707, 0, 0]
spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf',
        quat = initial_gripper_quat,
        pos=initial_gripper_pos,
        scale=1.0,
        merge_fixed_links=True,
        fixed = True
    ),
)

# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (1280, 960),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)
# Building the Scene
scene.build()

[Genesis] [17:27:57] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [17:27:57] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [17:27:57] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [17:27:57] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [17:27:57] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [17:27:57] [INFO] Scene <7cc5fdf> created.
[Genesis] [17:27:57] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <e4b5321>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [17:27:57] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <245c90e>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [17:27:58] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <6e0a354>, morph: <gs.morphs.URDF(file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [17:27:58] [INFO] Building scene <7cc5fdf>...
[Genesis] [17:28:05] [INFO] Compiling simulation kernels...
[Genesis] [17:28:32] [INFO] Building visualizer...
[Genesis] [17:28:33] [INFO] Viewer created. Resolution: 688×516, max_FPS: 600.


/home/nexus/miniconda3/lib/python3.12/site-packages/genesis/ext/pyrender/viewer.py


In [3]:
# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1]) 

# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )

In [4]:
def vector_to_euler(vec):
    vec = np.array(vec) / np.linalg.norm(vec)  # Normalize the vector
    reference = np.array([1, 0, 0])  # Reference vector
    
    if np.allclose(vec, reference):
        return np.array([0.0, 0.0, 0.0])
    
    axis = np.cross(reference, vec)
    angle = np.arccos(np.dot(reference, vec))
    
    if np.linalg.norm(axis) < 1e-6:
        axis = np.array([0, 0, 1])
    else:
        axis = axis / np.linalg.norm(axis)
    
    rotation = R.from_rotvec(angle * axis)
    print(rotation)
    euler_angles = rotation.as_euler('xyz')
    
    return euler_angles

In [5]:
def check_gripper_contacts():
    contacts = spot_gripper.get_contacts()
    
    if not contacts or 'position' not in contacts:
        return False
    
    num_contacts = len(contacts['position'])
    link_names = [l.name for l in scene.rigid_solver.links]
    
    bottle_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    ground_link_idx = link_names.index("plane_baselink") if "plane_baselink" in link_names else -1
    
    if bottle_link_idx == -1 or ground_link_idx == -1:
        return False
    
    bottle_contact = False
    ground_contact = False
    gripper_finger_links = ["arm_link_wr1", "arm_link_fngr"]
    
    for i in range(num_contacts):
        force_b = contacts['force_b'][i]
        norm = np.linalg.norm(force_b)
        normal = force_b / norm if norm > 1e-6 else np.zeros_like(force_b)
        link_a_idx = contacts['link_a'][i]
        link_b_idx = contacts['link_b'][i]
        link_a_name = link_names[link_a_idx] if link_a_idx < len(link_names) else 'Unknown'
        link_b_name = link_names[link_b_idx] if link_b_idx < len(link_names) else 'Unknown'
        
        if link_b_idx == bottle_link_idx and link_a_name in gripper_finger_links:
            bottle_contact = True
        elif link_a_idx == bottle_link_idx and link_b_name in gripper_finger_links:
            bottle_contact = True
        
        if (link_b_idx == ground_link_idx and link_a_name.startswith("arm_link_")) or \
           (link_a_idx == ground_link_idx and link_b_name.startswith("arm_link_")):
            ground_contact = True
    
    return bottle_contact and not ground_contact

In [6]:
def pixel_to_world(x, y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):
    """
    Convert pixel coordinates and depth to world coordinates
    
    Args:
        x: pixel x-coordinate (horizontal)
        y: pixel y-coordinate (vertical)
        depth: depth value at that pixel
        cam_pos: camera position (x, y, z)
        cam_lookat: point camera is looking at (x, y, z)
        fov: field of view in degrees
        img_width: width of the image in pixels
        img_height: height of the image in pixels
    
    Returns:
        world_coords: (x, y, z) in world coordinates
    """
    # Convert inputs to numpy arrays
    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)
    
    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)
    
    # Calculate camera up vector (assuming world up is (0,0,1))
    world_up = np.array(world_up)
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)
    
    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)
    
    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * y / (img_height - 1))  # Flip y-axis
    
    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)
    
    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0  # Negative z is forward in camera space
    
    # Create direction vector in camera space
    cam_space_dir = np.array([cam_space_x, cam_space_y, cam_space_z])
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir)
    
    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[0] + 
                cam_up * cam_space_dir[1] + 
                -cam_dir * cam_space_dir[2])
    world_dir = world_dir / np.linalg.norm(world_dir)
    
    # Calculate world position
    world_pos = cam_pos + world_dir * depth
    
    return world_pos

In [7]:
def check_bottle_contact():
    contacts = spot_gripper.get_contacts()
    if not contacts or 'position' not in contacts:
        return False
    
    num_contacts = len(contacts['position'])
    link_names = [l.name for l in scene.rigid_solver.links]
    
    bottle_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    if bottle_link_idx == -1:
        return False
    
    gripper_finger_links = ["arm_link_wr1", "arm_link_fngr"]
    
    for i in range(num_contacts):
        link_a_idx = contacts['link_a'][i]
        link_b_idx = contacts['link_b'][i]
        link_a_name = link_names[link_a_idx] if link_a_idx < len(link_names) else 'Unknown'
        link_b_name = link_names[link_b_idx] if link_b_idx < len(link_names) else 'Unknown'
        
        if link_b_idx == bottle_link_idx and link_a_name in gripper_finger_links:
            return True
        elif link_a_idx == bottle_link_idx and link_b_name in gripper_finger_links:
            return True
    
    return False

def perform_grasp(contact,  num_steps=10, pause_steps=10, max_gripper_pos=np.pi/2):

    current_qpos = spot_gripper.get_dofs_position()
    start_gripper_pos = current_qpos[finger_dof].item()
    gripper_step_size = (max_gripper_pos - start_gripper_pos) / num_steps
    
    for i in range(num_steps + 1):
        current_gripper_pos = start_gripper_pos + i * gripper_step_size
        target_qpos = current_qpos.clone()
        target_qpos[finger_dof] = current_gripper_pos
        spot_gripper.control_dofs_position(target_qpos)
        
        # Run simulation steps to stabilize and check contact
        for _ in range(pause_steps):
            scene.step()
        
        # Check for bottle contact
        if contact:
            break
    
    for _ in range(150):
        scene.step()
    

In [8]:
def is_in_xy_plane(quaternion, vector_normal, grasp_point, k_dist = 0.35):
    current_quat_array = quaternion.cpu().numpy()
    world_z_axis = np.array([0, 0, 1])
    world_z_quat = quaternion_from_euler(*vector_to_euler(-world_z_axis))
    validation = np.allclose(current_quat_array, world_z_quat, atol=1e-2)
    if validation == False:
        q_rot = quaternion_from_euler(np.pi/2, 0, 0)  
        gripper_quat = np.quaternion(*current_quat_array) * np.quaternion(*q_rot)
        gripper_quat_array = np.array([gripper_quat.w, gripper_quat.x, gripper_quat.y, gripper_quat.z])
        spot_gripper.set_quat(gripper_quat_array)
        spot_gripper.set_pos((k_dist/2) * vector_normal+grasp_point)  
        return True     
    else:
        spot_gripper.set_pos((k_dist/2) * vector_normal+grasp_point)  

        return False
for i in range(5):
    scene.step()

In [9]:
# Output directory
dataset_dir = Path("../water_bottle_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

# Verification directory
image_dir = Path("../img_verification_dataset")
image_dir.mkdir(parents=True, exist_ok=True)

# Number of samples
num_samples = 2

# Up direction reference
up = [0, 0, 1]

In [ ]:
for i in range(num_samples):
    ''' View setup'''
    view_id = f"view_{i}"
    view_dir = dataset_dir / view_id
    img_view_dir = image_dir / view_id
    view_dir.mkdir(parents=True, exist_ok=True)
    img_view_dir.mkdir(parents=True, exist_ok=True)
    
    ''' Radndomizing View '''
    
    #randomize camera position
    rand_cam_x = random.uniform(-0.5, 0.5)
    rand_cam_y = random.uniform(-0.5, 0.5)
    rand_cam_z = random.uniform(0.16, 0.6)

    # randomize camera lookat
    rand_lookat_x = random.uniform(-0.03, 0.03)
    rand_lookat_y = random.uniform(-0.03, 0.03)
    rand_lookat_z = random.uniform(0, 0.09)   
    
    # change camera position
    cam.set_pose(
        pos = (rand_cam_x,rand_cam_y,rand_cam_z),
        lookat= (rand_lookat_x, rand_lookat_y, rand_lookat_z)
        )
    
    # Get RGB-D data
    for i in range(5):
        scene.step()
        render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False, )

    rgb, depth, seg, normal_map = render_output
    cylinder_pixel_coords = np.where(np.array(seg) == 1)
    cylinder_pixel_coords=np.transpose(cylinder_pixel_coords)

    # Save data
    np.save(view_dir / "rgb.npy", rgb)
    np.save(view_dir / "depth.npy", depth)
    np.save(view_dir / "segmentation.npy", seg)
    np.save(view_dir / "normal.npy", normal_map)
    
    # Save PNGs for visual verification
    cv2.imwrite(str(img_view_dir / "rgb.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    random_idx = random.randint(0, len(cylinder_pixel_coords) - 1)
    random_y, random_x = cylinder_pixel_coords[random_idx]
    specific_point_depth = depth[random_y, random_x]
    specific_point_normal = normal_map[random_y, random_x]

    vector_normal = 2 * (specific_point_normal - 127.5) / 255

    # Camera common parameters
    cam_lookat = [rand_lookat_x, rand_lookat_y, rand_lookat_z]
    cam_pos = [rand_cam_x,rand_cam_y,rand_cam_z]
    img_width, img_height, fov = 1280, 960, 30
    
    # Grasp parameters
    grasp_point = pixel_to_world(random_x, random_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up)
    quartenion = quaternion_from_euler(* vector_to_euler(-vector_normal))
    position = grasp_point + 0.5*vector_normal
    
    # Approach gripper from object
    spot_gripper.set_pos(position)
    spot_gripper.set_quat(quartenion)
    current_pos = spot_gripper.get_pos()
    current_quat = spot_gripper.get_quat()

    is_in_xy_plane(current_quat, vector_normal, grasp_point)
    for _ in range(200):
        scene.step()

    contacts = check_bottle_contact()
    perform_grasp(contacts)
    for _ in range(200):
        scene.step()
            
    final_contacts = check_gripper_contacts()
    for _ in range(200):
        scene.step()

    # Store metadata
    data_metadata = []
    data_metadata.append({
        "view_id": view_id,
        "camera_pos": cam_pos,
        "camera_lookat": cam_lookat,
        "final_contacts": final_contacts,
    })
    # Save metadata
    with open( view_dir / "metadata.json", "w") as f:
        json.dump(data_metadata, f, indent=4)
    
    with open(view_dir / "grasp_success.txt", "w") as f:
        f.write(str(final_contacts))

    # Back gripper to inital location
    spot_gripper.set_pos(initial_gripper_pos)
    spot_gripper.set_quat(initial_gripper_quat)
    for _ in range(200):
        scene.step()

[Genesis] [17:28:33] [WARNING] Base link is fixed. Overriding base link pose.


[Genesis] [17:28:33] [WARNING] Base link is fixed. Overriding base link pose.
/tmp/ipykernel_22112/3605326670.py:67: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  is_in_xy_plane(current_quat, vector_normal, grasp_point)
[Genesis] [17:28:33] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:33] [WARNING] Base link is fixed. Overriding base link pose.


[Genesis] [17:28:35] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:35] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:36] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:36] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:36] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:36] [WARNING] Base link is fixed. Overriding base link pose.


[Genesis] [17:28:38] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:28:38] [WARNING] Base link is fixed. Overriding base link pose.
